# Hotel Doc QnA RAG Chat Bot using LangChain and ChromaDB

## 1. Loading Modules

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

from langchain_groq import ChatGroq

from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_community.chat_message_histories import SQLChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_core.messages import AIMessage
from langchain_core.messages import HumanMessage

from IPython.display import display, Markdown

## 2. Load API Keys and Set Up Global Configurations

In [2]:
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [3]:
ollama_embed_model = 'nomic-embed-text:latest'

embed_ollama = OllamaEmbeddings(model=ollama_embed_model)

embed_ollama

OllamaEmbeddings(model='nomic-embed-text:latest', base_url=None, client_kwargs={})

In [4]:
llm_model = "llama-3.1-8b-instant"

llm = ChatGroq(model=llm_model, temperature=0.2)

llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000020D9ED351B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020D9ED35E70>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

## 3. App Input Data Loading

In [5]:
file = r'./data/Ocean_Breeze_Resort.pdf'

loader = PyPDFLoader(file_path=file)       # password=''
docs = loader.load()

print(f'Loaded File: "{file}"')
print(f'No of Documents Created: {len(docs)}')

Loaded File: "./data/Ocean_Breeze_Resort.pdf"
No of Documents Created: 4


In [6]:
print(f'Document Metadata: {docs[0].metadata} \n')

print('Sample Document Data: ')
docs[0].page_content

Document Metadata: {'source': './data/Ocean_Breeze_Resort.pdf', 'page': 0} 

Sample Document Data: 


'Hotel Name: Ocean Breeze Resort  \nLocation: Tranquil Bay, Coastal City, [State]  \nWebsite: www.oceanbreezeresort.com  [fake website ] \nEmail: [email address removed]  \nDescription:  Nestled along the pristine shores of Tranquil Bay, Ocean Breeze Resort offers a serene escape with \nbreathtaking ocean views. Our luxurious accommodations feature modern amenities and private balconies \noverlooking the crystal -clear waters. Indulge in delectable cuisine at our beachfront restaurant, unwind at our \noutdoor pool and spa, or explore the vibra nt local culture. Experience the perfect blend of relaxation and adventure \nat Ocean Breeze Resort.  \nServices  \nOcean Breeze Resort offers a wide range of services to ensure a comfortable and enjoyable stay. Our friendly and \nattentive staff is dedicated to providing exceptional service.  \n• Concierge Services:  Our concierge can assist you with making reservations, arranging transportation, and \nproviding local recommendations.  \n• Busin

## 4. Data Pre-Processing

In [7]:
# create_documents() : text
# split_text() : text 
# split_documents() : documents

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

docs_splits = text_splitter.split_documents(docs)

print(f'No of chunks created: {len(docs_splits)}')

No of chunks created: 18


In [8]:
print(f'Document Metadata: {docs_splits[0].metadata} \n')

print('Sample Chunk Data: ')
docs_splits[0].page_content

Document Metadata: {'source': './data/Ocean_Breeze_Resort.pdf', 'page': 0} 

Sample Chunk Data: 


'Hotel Name: Ocean Breeze Resort  \nLocation: Tranquil Bay, Coastal City, [State]  \nWebsite: www.oceanbreezeresort.com  [fake website ] \nEmail: [email address removed]  \nDescription:  Nestled along the pristine shores of Tranquil Bay, Ocean Breeze Resort offers a serene escape with \nbreathtaking ocean views. Our luxurious accommodations feature modern amenities and private balconies \noverlooking the crystal -clear waters. Indulge in delectable cuisine at our beachfront restaurant, unwind at our \noutdoor pool and spa, or explore the vibra nt local culture. Experience the perfect blend of relaxation and adventure \nat Ocean Breeze Resort.  \nServices  \nOcean Breeze Resort offers a wide range of services to ensure a comfortable and enjoyable stay. Our friendly and \nattentive staff is dedicated to providing exceptional service.  \n• Concierge Services:  Our concierge can assist you with making reservations, arranging transportation, and \nproviding local recommendations.'

## 5. Vector Store (ChromaDB)

### 5.1 Create

In [9]:
# vector_store = Chroma.from_documents(documents=docs_splits, persist_directory='./chroma_db',
#                                      collection_name='obr', embedding=embed_ollama)
# vector_store

### 5.2 Load

In [10]:
load_vector_store = Chroma(persist_directory='./chroma_db', collection_name='obr', embedding_function=embed_ollama)

load_vector_store

### 5.3 Delete

In [11]:
# load_vector_store.delete_collection()

## 6. Retreiver Response

### 6.1 Prompt Template

In [12]:
sys_template = '''
You are a polite, helpful and expert AI assistant in the domain of hotel and customer management, and your name is "Karen".

CONTEXT:
--------
{context}

Your job is to answer the user queries related to the hotel named "Ocean Breeze Resort" only. You can use emojis to answer!
Introduce yourself, only in the start of conversation or when asked!
You should never talk about yourself such as training, documents, sources, architecture, last updates, or who you are in depth!
You should sound confident and bold in your answers, without any hesitation!
For any disclaimers, show them at the start of the answers only!
Keep your answer short and precise based on facts as much as possible regarding the query – do not hallucinate features!

Provide a very short and suitable excuse for all non-related queries without any further suggestions, recommendations, or guidance!
'''

### 6.2 Chain to Retrieve Relevant Documents

In [13]:
retriever = load_vector_store.as_retriever(search_kwargs={"k": 1})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


retrieved_docs = retriever | format_docs

retrieved_docs

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000020D930F2E30>, search_kwargs={'k': 1})
| RunnableLambda(format_docs)

## 7. ChatBot Functionality

### 7.1 Chat History Config

In [14]:
chat_message_history = SQLChatMessageHistory(session_id="test_session_id", connection="sqlite:///hotel_cb_chats.db")

chat_message_history.add_ai_message("Hi, Welcome to Ocean Breeze Resort Hotel.")

chat_message_history

### 7.2 Final Prompt for LLM

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", sys_template),
        MessagesPlaceholder(variable_name="history"),
        ("human","{question}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['context', 'history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotat

### 7.3 LCEL Chain

In [16]:
chain = prompt | llm | StrOutputParser()

chain

ChatPromptTemplate(input_variables=['context', 'history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotat

### 7.4 Runnable Message History Chain and Config

In [17]:
chain_with_history = RunnableWithMessageHistory(chain, 
                lambda session_id: SQLChatMessageHistory(session_id=session_id, connection="sqlite:///hotel_cb_chats.db"),
                input_messages_key="question", 
                history_messages_key="history")

config = {"configurable": {"session_id": "test_session_id"}}

chain_with_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function <lambda> at 0x0000020D930DF520>, input_messages_key='question', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

## 8. LLM Response

### 8.1 RAG QnA

In [18]:
question = 'List the various services offered by the hotel. Please, categorize them also under suitable heads.'

response = chain_with_history.invoke({"context":retrieved_docs, "question": question}, config=config)

display(Markdown(response))

**Services Offered at Ocean Breeze Resort Hotel 🏨**

**Accommodation Services 🛏️**

- Luxurious rooms and suites
- Family rooms and interconnected rooms
- Accessible rooms for guests with disabilities
- Room service available

**Dining Services 🍴**

- Ocean Breeze Restaurant (serving international cuisine)
- Beachside Bar & Grill (serving snacks and light meals)
- In-room dining
- Room service

**Recreational Services 🏖️**

- Private beach access
- Outdoor pool
- Fitness center
- Beach volleyball and other games
- Water sports (available at an extra cost)

**Business Services 📊**

- Meeting and event spaces
- Conference facilities
- Business center
- Wi-Fi connectivity throughout the hotel

**Other Services 🚿**

- Tour and travel desk
- Laundry and dry cleaning services
- Spa and wellness center
- Concierge services

### 8.2 Chat History in DB

In [19]:
for msg in chat_message_history.messages:
    if isinstance(msg, HumanMessage):
        display(Markdown(f'***USER***: {msg.content}'))
    else:
        display(Markdown(f'***AI***: {msg.content}'))
        print('\n')

***AI***: Hi, Welcome to Ocean Breeze Resort Hotel.

***USER***: List the various services offered by the hotel. Please, categorize them also under suitable heads.

***AI***: **Services Offered at Ocean Breeze Resort Hotel 🏨**

**Accommodation Services 🛏️**

- Luxurious rooms and suites
- Family rooms and interconnected rooms
- Accessible rooms for guests with disabilities
- Room service available

**Dining Services 🍴**

- Ocean Breeze Restaurant (serving international cuisine)
- Beachside Bar & Grill (serving snacks and light meals)
- In-room dining
- Room service

**Recreational Services 🏖️**

- Private beach access
- Outdoor pool
- Fitness center
- Beach volleyball and other games
- Water sports (available at an extra cost)

**Business Services 📊**

- Meeting and event spaces
- Conference facilities
- Business center
- Wi-Fi connectivity throughout the hotel

**Other Services 🚿**

- Tour and travel desk
- Laundry and dry cleaning services
- Spa and wellness center
- Concierge services

### 8.3 Clear Message History

### 8.4 Streaming Response

## 9. Questions by User to the Chat Bot